#**Proceso de ETL (Extracción, Transformación y Carga)**

Dataset - Ventas de Videojuegos - Kaggle

**1. Extracción de Datos**

In [3]:
import pandas as pd

df = pd.read_csv('vgsales.csv')
print (df.head())


   Rank                      Name Platform    Year         Genre Publisher  \
0     1                Wii Sports      Wii  2006.0        Sports  Nintendo   
1     2         Super Mario Bros.      NES  1985.0      Platform  Nintendo   
2     3            Mario Kart Wii      Wii  2008.0        Racing  Nintendo   
3     4         Wii Sports Resort      Wii  2009.0        Sports  Nintendo   
4     5  Pokemon Red/Pokemon Blue       GB  1996.0  Role-Playing  Nintendo   

   NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales  
0     41.49     29.02      3.77         8.46         82.74  
1     29.08      3.58      6.81         0.77         40.24  
2     15.85     12.88      3.79         3.31         35.82  
3     15.75     11.01      3.28         2.96         33.00  
4     11.27      8.89     10.22         1.00         31.37  


**2. Transformación de datos**

In [4]:
# Eliminar duplicados
df = df.drop_duplicates()

# Rellenar valores nulos
df = df.fillna("Unknown")

# Convertir año a entero (algunos vienen como float o NaN)
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').fillna(0).astype(int)

# Crear una métrica: ventas totales por género
sales_by_genre = df.groupby("Genre")["Global_Sales"].sum().reset_index()
print(sales_by_genre)


           Genre  Global_Sales
0         Action       1751.18
1      Adventure        239.04
2       Fighting        448.91
3           Misc        809.96
4       Platform        831.37
5         Puzzle        244.95
6         Racing        732.04
7   Role-Playing        927.37
8        Shooter       1037.37
9     Simulation        392.20
10        Sports       1330.93
11      Strategy        175.12


**Calcular el top 5 publishers por ventas globales:**

In [8]:
top_publishers = df.groupby("Publisher")["Global_Sales"].sum().nlargest(10).reset_index()
print(top_publishers)


                      Publisher  Global_Sales
0                      Nintendo       1786.56
1               Electronic Arts       1110.32
2                    Activision        727.46
3   Sony Computer Entertainment        607.50
4                       Ubisoft        474.72
5          Take-Two Interactive        399.54
6                           THQ        340.77
7  Konami Digital Entertainment        283.64
8                          Sega        272.99
9            Namco Bandai Games        254.09


# Top Ventas en Japon

In [11]:
japan_sales = df.groupby('Name')['JP_Sales'].sum().nlargest(10).reset_index()
print(japan_sales)
jp_genre_sales = df.groupby('Genre')['JP_Sales'].sum().nlargest(10).reset_index()
print('\n',jp_genre_sales)

                                 Name  JP_Sales
0            Pokemon Red/Pokemon Blue     10.22
1         Pokemon Gold/Pokemon Silver      7.20
2                   Super Mario Bros.      6.96
3               New Super Mario Bros.      6.50
4       Pokemon Diamond/Pokemon Pearl      6.04
5                              Tetris      6.03
6         Pokemon Black/Pokemon White      5.65
7  Dragon Quest VII: Warriors of Eden      5.40
8       Pokemon Ruby/Pokemon Sapphire      5.38
9         Animal Crossing: Wild World      5.33

           Genre  JP_Sales
0  Role-Playing    352.31
1        Action    159.95
2        Sports    135.37
3      Platform    130.77
4          Misc    107.76
5      Fighting     87.35
6    Simulation     63.70
7        Puzzle     57.31
8        Racing     56.69
9     Adventure     52.07


**3. Carga de los datos**


In [11]:
import sqlite3

conn = sqlite3.connect("videogames.db")
df.to_sql("vgsales_clean", conn, if_exists="replace", index=False)
sales_by_genre.to_sql("sales_by_genre", conn, if_exists="replace", index=False)
top_publishers.to_sql("top_publishers", conn, if_exists="replace", index=False)
conn.close()

print ("Datos cargados en la base de datos.")
#

Datos cargados en la base de datos.


# Visualización de los datos en dashboard para mejor interpretación

In [12]:
try:
    import streamlit as st
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit"])
    import streamlit as st
import sqlite3
import pandas as pd

# Conectar a la base
conn = sqlite3.connect("videogames.db")

# Cargar tabla
df = pd.read_sql("SELECT * FROM vgsales_clean", conn)

# Dashboard
st.title("Ventas de Videojuegos")
st.bar_chart(df.groupby("Genre")["Global_Sales"].sum())
st.line_chart(df.groupby("Year")["Global_Sales"].sum())


2026-05-25 20:43:12.202 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 20:43:12.524 
  command:

    streamlit run C:\Users\kenne\AppData\Roaming\Python\Python314\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-25 20:43:12.525 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 20:43:12.525 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 20:43:13.469 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 20:43:13.471 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 20:43:13.473 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 20:43:13.530 Thre

DeltaGenerator()